# Pandas Tutorial: Open Data Portal der Stadt Zürich

Dieses Tutorial zeigt, wie man mit **pandas** und der **CKAN-API** des Open Data Portals der Stadt Zürich (`data.stadt-zuerich.ch`) arbeitet.

## API-Übersicht

Das Portal basiert auf [CKAN](https://ckan.org/) und stellt über 900 Datensätze unter der CC0-Lizenz bereit. Die REST-API ist unter folgendem Basis-URL erreichbar:

```
https://data.stadt-zuerich.ch/api/3/action/
```

### Wichtige Endpunkte

| Endpunkt | Beschreibung |
|---|---|
| `package_search` | Datensätze suchen |
| `package_show` | Metadaten eines Datensatzes abrufen |
| `datastore_search` | Tabellarische Daten filtern & abfragen |
| `datastore_search_sql` | SQL-Abfragen auf Datensätzen |
| `group_list` | Alle Datenkategorien auflisten |
| `resource_show` | Details einer Ressource (Datei) |

---

## 1. Setup & Imports

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 60)

# Basis-URL der CKAN-API
BASE_URL = "https://data.stadt-zuerich.ch/api/3/action"

def ckan_get(endpoint: str, params: dict = None) -> dict:
    """Hilfsfunktion: GET-Anfrage an die CKAN-API, gibt das 'result'-Feld zurück."""
    url = f"{BASE_URL}/{endpoint}"
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()
    if not data.get("success"):
        raise ValueError(f"API-Fehler: {data.get('error')}")
    return data["result"]

print("Setup abgeschlossen.")

---
## 2. Katalog erkunden: Kategorien & Statistiken

Zuerst verschaffen wir uns einen Überblick über die verfügbaren Datenkategorien.

In [ ]:
# Alle Kategorien (Gruppen) mit Datensatz-Anzahl abrufen
groups_raw = ckan_get("group_list", params={"all_fields": True, "include_dataset_count": True})

# In DataFrame umwandeln
df_groups = pd.DataFrame(groups_raw)[["name", "display_name", "package_count"]]
df_groups.columns = ["id", "Kategorie", "Anzahl Datensätze"]
df_groups = df_groups.sort_values("Anzahl Datensätze", ascending=False).reset_index(drop=True)

print(f"Anzahl Kategorien: {len(df_groups)}")
df_groups

In [ ]:
# Balkendiagramm der Kategorien
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(df_groups["Kategorie"][::-1], df_groups["Anzahl Datensätze"][::-1], color="steelblue")
ax.set_xlabel("Anzahl Datensätze")
ax.set_title("Open Data Stadt Zürich – Datensätze pro Kategorie")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

---
## 3. Datensätze suchen

Mit `package_search` können wir die Volltextsuche nutzen und nach Themen filtern.

In [ ]:
def search_datasets(query: str, group: str = None, rows: int = 10) -> pd.DataFrame:
    """Durchsucht den Katalog und gibt ein DataFrame mit Ergebnissen zurück."""
    params = {
        "q": query,
        "rows": rows,
        "sort": "score desc",
    }
    if group:
        params["fq"] = f'groups:{group}'

    result = ckan_get("package_search", params=params)
    datasets = result["results"]

    rows_data = []
    for ds in datasets:
        rows_data.append({
            "id": ds["id"],
            "name": ds["name"],
            "Titel": ds.get("title", ""),
            "Ressourcen": len(ds.get("resources", [])),
            "Formate": ", ".join({r["format"] for r in ds.get("resources", []) if r.get("format")}),
            "Aktualisiert": ds.get("metadata_modified", "")[:10],
        })
    return pd.DataFrame(rows_data)


# Suche nach Bevölkerungsdaten
df_results = search_datasets("Bevölkerung Quartier", rows=8)
df_results[["Titel", "name", "Ressourcen", "Formate", "Aktualisiert"]]

---
## 4. Metadaten eines Datensatzes abrufen

Mit `package_show` rufen wir vollständige Metadaten inklusive aller Ressourcen-URLs ab.

In [ ]:
# Metadaten des Datensatzes "Bevölkerung nach Quartier" abrufen
DATASET_ID = "bev390od3903"  # Bevölkerung nach Heimat, Geschlecht, Quartier

meta = ckan_get("package_show", params={"id": DATASET_ID})

print(f"Titel    : {meta['title']}")
print(f"Lizenz   : {meta.get('license_title', 'n/a')}")
print(f"Autor    : {meta.get('author', 'n/a')}")
print(f"Intervall: {meta.get('accrual_periodicity', 'n/a')}")
print(f"Notizen  : {meta.get('notes', '')[:200]}...")
print()

# Ressourcen (Dateien/Endpunkte) des Datensatzes
df_resources = pd.DataFrame(meta["resources"])[["id", "name", "format", "url"]]
df_resources

---
## 5. Daten direkt per CSV-URL laden

Viele Datensätze stellen CSV-Dateien bereit, die sich direkt mit `pd.read_csv()` laden lassen.

In [ ]:
# Erste CSV-Ressource aus den Metadaten auswählen
csv_resources = df_resources[df_resources["format"].str.upper() == "CSV"]
csv_url = csv_resources.iloc[0]["url"]
print(f"Lade CSV von: {csv_url}")

df_bev = pd.read_csv(csv_url)
print(f"\nShape: {df_bev.shape}")
df_bev.head()

In [ ]:
# Datentypen & fehlende Werte prüfen
df_bev.info()

In [ ]:
# Statistische Zusammenfassung der numerischen Spalten
df_bev.describe()

---
## 6. Daten über den CKAN DataStore abfragen

Der DataStore ermöglicht gefilterte Abfragen direkt über die API — ohne die gesamte Datei herunterzuladen.

### 6.1 Einfache Filter-Abfrage (`datastore_search`)

In [ ]:
def datastore_search(
    resource_id: str,
    filters: dict = None,
    q: str = None,
    limit: int = 100,
    fields: list = None,
    sort: str = None,
) -> pd.DataFrame:
    """Fragt den CKAN DataStore ab und gibt ein pandas DataFrame zurück."""
    params = {"resource_id": resource_id, "limit": limit}
    if filters:
        import json
        params["filters"] = json.dumps(filters)
    if q:
        params["q"] = q
    if fields:
        params["fields"] = ",".join(fields)
    if sort:
        params["sort"] = sort

    result = ckan_get("datastore_search", params=params)
    return pd.DataFrame(result["records"])


# Resource-ID aus den Metadaten (erste CSV-Ressource)
RESOURCE_ID = csv_resources.iloc[0]["id"]
print(f"Resource-ID: {RESOURCE_ID}")

# Daten für ein bestimmtes Jahr und Geschlecht laden
df_query = datastore_search(
    resource_id=RESOURCE_ID,
    limit=50,
    sort="StichtagDatJahr desc",
)
df_query.head(10)

### 6.2 SQL-Abfrage (`datastore_search_sql`)

Für komplexere Auswertungen unterstützt der DataStore SQL-ähnliche Abfragen.

In [ ]:
def datastore_sql(sql: str) -> pd.DataFrame:
    """Führt eine SQL-Abfrage auf dem CKAN DataStore aus."""
    result = ckan_get("datastore_search_sql", params={"sql": sql})
    return pd.DataFrame(result["records"])


# Bevölkerungssumme pro Jahr aggregieren
sql = f"""
    SELECT
        "StichtagDatJahr" AS Jahr,
        SUM("AnzBestWir") AS Bevoelkerung
    FROM \"{RESOURCE_ID}\"
    GROUP BY "StichtagDatJahr"
    ORDER BY "StichtagDatJahr"
"""

df_yearly = datastore_sql(sql)
df_yearly["Jahr"] = pd.to_numeric(df_yearly["Jahr"])
df_yearly["Bevoelkerung"] = pd.to_numeric(df_yearly["Bevoelkerung"])
df_yearly

In [ ]:
# Bevölkerungsentwicklung visualisieren
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_yearly["Jahr"], df_yearly["Bevoelkerung"], marker="o", linewidth=2, color="steelblue")
ax.fill_between(df_yearly["Jahr"], df_yearly["Bevoelkerung"], alpha=0.15, color="steelblue")
ax.set_title("Bevölkerungsentwicklung in Zürich")
ax.set_xlabel("Jahr")
ax.set_ylabel("Bevölkerung")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Datenanalyse mit pandas

Wir wenden typische pandas-Operationen auf die Bevölkerungsdaten an.

In [ ]:
# Vollständigen Datensatz laden (alle Jahre, alle Quartiere)
# Spalten vorab per DataStore-API erkunden
fields_result = ckan_get("datastore_search", params={"resource_id": RESOURCE_ID, "limit": 0})
available_fields = [f["id"] for f in fields_result["fields"] if f["id"] != "_id"]
print("Verfügbare Felder:", available_fields)

In [ ]:
# Gesamten Datensatz laden (paginiert, falls nötig)
def load_full_dataset(resource_id: str, batch_size: int = 1000) -> pd.DataFrame:
    """Lädt alle Datensätze aus dem DataStore (mit automatischer Paginierung)."""
    offset = 0
    frames = []

    # Gesamtanzahl abrufen
    total = ckan_get("datastore_search", params={"resource_id": resource_id, "limit": 0})["total"]
    print(f"Gesamt: {total:,} Datensätze")

    while offset < total:
        result = ckan_get("datastore_search", params={
            "resource_id": resource_id,
            "limit": batch_size,
            "offset": offset,
        })
        batch = pd.DataFrame(result["records"])
        frames.append(batch)
        offset += batch_size
        print(f"  Geladen: {min(offset, total):,} / {total:,}", end="\r")

    print()
    return pd.concat(frames, ignore_index=True)


df = load_full_dataset(RESOURCE_ID)
df.head()

In [ ]:
# Datentypen bereinigen
num_cols = ["StichtagDatJahr", "AnzBestWir"]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df.dtypes

### 7.1 Aktuellste Bevölkerung pro Quartier

In [ ]:
# Neuestes Jahr im Datensatz
latest_year = df["StichtagDatJahr"].max()
print(f"Aktuellstes Jahr: {latest_year}")

# Bevölkerung pro Quartier für das neueste Jahr
df_latest = (
    df[df["StichtagDatJahr"] == latest_year]
    .groupby("QuarLang", as_index=False)["AnzBestWir"]
    .sum()
    .sort_values("AnzBestWir", ascending=False)
    .rename(columns={"QuarLang": "Quartier", "AnzBestWir": "Bevölkerung"})
)

df_latest.head(10)

In [ ]:
# Top-10 bevölkerungsreichste Quartiere
top10 = df_latest.head(10)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top10["Quartier"][::-1], top10["Bevölkerung"][::-1], color="coral")
ax.bar_label(bars, fmt="{:,.0f}", padding=4, fontsize=9)
ax.set_xlabel("Bevölkerung")
ax.set_title(f"Top-10 Quartiere nach Bevölkerung ({int(latest_year)})")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.show()

### 7.2 Bevölkerungswachstum nach Quartier (Pivot-Tabelle)

In [ ]:
# Pivot: Quartier × Jahr
df_pivot = (
    df.groupby(["QuarLang", "StichtagDatJahr"])["AnzBestWir"]
    .sum()
    .unstack("StichtagDatJahr")
)

# Wachstum in % gegenüber erstem verfügbaren Jahr
first_year = df["StichtagDatJahr"].min()
last_year = df["StichtagDatJahr"].max()

df_growth = pd.DataFrame({
    "Bevölkerung Start": df_pivot[first_year],
    "Bevölkerung Ende": df_pivot[last_year],
})
df_growth["Wachstum %"] = (
    (df_growth["Bevölkerung Ende"] - df_growth["Bevölkerung Start"])
    / df_growth["Bevölkerung Start"] * 100
).round(1)
df_growth = df_growth.sort_values("Wachstum %", ascending=False)

print(f"Wachstum {int(first_year)}–{int(last_year)}:")
df_growth.head(10)

### 7.3 Geschlechterverteilung (falls Spalte vorhanden)

In [ ]:
# Prüfen, ob Geschlecht-Spalte vorhanden
sex_col = next((c for c in df.columns if "sex" in c.lower() or "geschl" in c.lower()), None)

if sex_col:
    df_sex = (
        df[df["StichtagDatJahr"] == latest_year]
        .groupby(sex_col)["AnzBestWir"]
        .sum()
    )
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.pie(df_sex, labels=df_sex.index, autopct="%1.1f%%", colors=["steelblue", "coral"])
    ax.set_title(f"Geschlechterverteilung {int(latest_year)}")
    plt.tight_layout()
    plt.show()
else:
    print("Keine Geschlecht-Spalte im Datensatz gefunden.")
    print("Verfügbare Spalten:", df.columns.tolist())

---
## 8. Zweites Beispiel: Weitere Datensätze erkunden

Die gleichen Techniken lassen sich auf jeden anderen Datensatz anwenden.
Als Beispiel suchen wir nach Mobilitätsdaten.

In [ ]:
# Mobilitätsdaten suchen
df_mobil = search_datasets("Verkehr Velo", group="mobilitat", rows=5)
df_mobil[["Titel", "name", "Formate"]]

In [ ]:
# Allgemeine Funktion: CSV-Ressource eines Datensatzes laden
def load_dataset_csv(dataset_name: str) -> pd.DataFrame:
    """Lädt die erste CSV-Ressource eines Datensatzes als DataFrame."""
    meta = ckan_get("package_show", params={"id": dataset_name})
    csv_res = [
        r for r in meta["resources"]
        if r.get("format", "").upper() == "CSV"
    ]
    if not csv_res:
        raise ValueError(f"Keine CSV-Ressource für '{dataset_name}' gefunden.")
    url = csv_res[0]["url"]
    print(f"Lade: {url}")
    return pd.read_csv(url)


# Beispiel-Aufruf (Dataset-Name aus der Suche oben anpassen)
# df_velo = load_dataset_csv("<dataset-name-hier>")
# df_velo.head()
print("Funktion load_dataset_csv() bereit.")
print("Rufe sie mit dem 'name'-Wert aus der Suchresultat-Tabelle auf.")

---
## 9. SPARQL: Linked Data abfragen

Das Portal bietet auch einen SPARQL-Endpunkt für Linked Open Data.

In [ ]:
SPARQL_URL = "https://ld.stadt-zuerich.ch/query"

def sparql_query(query: str) -> pd.DataFrame:
    """Führt eine SPARQL-Abfrage aus und gibt ein DataFrame zurück."""
    headers = {"Accept": "application/sparql-results+json"}
    response = requests.get(SPARQL_URL, params={"query": query}, headers=headers, timeout=30)
    response.raise_for_status()
    data = response.json()
    bindings = data["results"]["bindings"]
    rows = [{k: v["value"] for k, v in row.items()} for row in bindings]
    return pd.DataFrame(rows)


# Beispiel: Bevölkerungszahlen via SPARQL
query = """
PREFIX qb: <http://purl.org/linked-data/cube#>
PREFIX dataset: <https://ld.stadt-zuerich.ch/statistics/dataset/>
PREFIX measure: <https://ld.stadt-zuerich.ch/statistics/measure/>
PREFIX dimension: <https://ld.stadt-zuerich.ch/statistics/property/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?Jahr ?Bevoelkerung
WHERE {
  ?obs a qb:Observation ;
       qb:dataSet dataset:BEW-RAUM-ZEIT ;
       dimension:ZEIT ?Zeit ;
       measure:BEW ?Bevoelkerung .
  BIND(year(?Zeit) AS ?Jahr)
  FILTER(?Jahr >= 2010)
}
ORDER BY ?Jahr
LIMIT 20
"""

try:
    df_sparql = sparql_query(query)
    print(f"Ergebnis: {len(df_sparql)} Zeilen")
    df_sparql
except Exception as e:
    print(f"SPARQL-Abfrage fehlgeschlagen: {e}")
    print("Der SPARQL-Endpunkt ist unter https://ld.stadt-zuerich.ch/sparql erreichbar.")

---
## 10. Zusammenfassung

### Was wir gelernt haben

| Aufgabe | Methode |
|---|---|
| Katalog erkunden | `group_list` → `pd.DataFrame()` |
| Datensätze suchen | `package_search` mit `q` und `fq` |
| Metadaten abrufen | `package_show` |
| CSV direkt laden | `pd.read_csv(url)` |
| Gefilterte Abfrage | `datastore_search` mit `filters` / `q` |
| SQL-Aggregation | `datastore_search_sql` |
| Paginierung | Loop mit `offset` |
| Linked Data | SPARQL-Endpunkt |

### Nützliche Links

- **Portal**: https://data.stadt-zuerich.ch
- **CKAN-API-Doku**: https://docs.ckan.org/en/latest/api/
- **SPARQL-Endpunkt**: https://ld.stadt-zuerich.ch/sparql
- **Datensatz-Suche**: https://data.stadt-zuerich.ch/dataset

### Tipps

- Alle Daten stehen unter **CC0** – keine Lizenzpflichten.
- Große Datensätze über Paginierung (`offset`) oder SQL-Aggregation abrufen.
- `datastore_search_sql` eignet sich für Aggregationen direkt auf dem Server.
- Resource-IDs sind stabil und können direkt gebookmarkt werden.